In [1]:
import tensorflow as tf, keras
print("TF:", tf.__version__, "Keras:", keras.__version__)

TF: 2.19.0 Keras: 3.10.0


In [2]:
from keras import layers
from keras.models import Model

from keras.layers import Input, RNN
from keras.layers import Layer, LayerNormalization
from keras.layers import Dense, Dropout

import numpy as np

In [3]:
def get_add(n_data, seq_len, seed=42):
    if seed is not None:
        np.random.seed(seed)

    x = np.zeros((n_data, seq_len, 2), dtype=np.float32)
    x[:, :, 0] = np.random.uniform(0.0, 1.0, size=(n_data, seq_len)).astype(np.float32)

    half = seq_len // 2
    idx_first = np.random.randint(0, half, size=n_data)
    idx_second = np.random.randint(half, seq_len, size=n_data)

    rows = np.arange(n_data)
    x[rows, idx_first, 1] = 1.0
    x[rows, idx_second, 1] = 1.0

    y = (x[:, :, 0] * x[:, :, 1]).sum(axis=1, keepdims=True).astype(np.float32)

    n_train, n_valid, n_test = 100000, 5000, 35000
    n_train = min(n_train, n_data)
    n_valid = min(n_valid, n_data - n_train)
    n_test  = min(n_test,  n_data - n_train - n_valid)

    x_train, y_train = x[:n_train], y[:n_train]
    x_valid, y_valid = x[n_train:n_train + n_valid], y[n_train:n_train + n_valid]
    x_test,  y_test  = x[n_train + n_valid:n_train + n_valid + n_test], y[n_train + n_valid:n_train + n_valid + n_test]

    return x_train, y_train, x_valid, y_valid, x_test, y_test

In [4]:
def chrono_init_cri(t_max, num_units, flag):
    def _initializer(shape, dtype=None):
        dtype = dtype or tf.float32
        target_shape = shape or (num_units,)
        rnd = tf.random.uniform(target_shape, minval=1.0, maxval=float(t_max), dtype=dtype, seed=42)
        init_values = tf.math.log(rnd)
        return init_values if flag == 1 else -init_values
    return _initializer

#TODO: use the previous method! Leave it as is for now for testing purposes.
def mchrono_init_cri(t_max, num_units):
    def _initializer(shape, dtype=None):
        dtype = dtype or tf.float32
        target_shape = shape or (num_units,)
        rnd = tf.random.uniform(target_shape, minval=1.0, maxval=float(t_max), dtype=dtype, seed=42)
        return -tf.math.log(rnd)
    return _initializer

In [ ]:
#shared statistics
@tf.keras.utils.register_keras_serializable()
class CILN(tf.keras.layers.Layer):
    def __init__(self, units, t_max, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.t_max = t_max
        self.state_size = [units, units]  

        self.ln_gates = LayerNormalization(
            center=False,
            scale=True,
            name="ln_gates"
        )
        self.ln_out = LayerNormalization(
            center=False,
            scale=True,
            name="ln_out"
        )

    def build(self, input_shape):
        input_dim = int(input_shape[-1])
        concat_dim = input_dim + self.units

        self.W = self.add_weight(
            name="W_xh",
            shape=(concat_dim, 4 * self.units),
            initializer="glorot_uniform",
            trainable=True,

        )

        self.bias_f = self.add_weight(
            name="bias_f",
            shape=(self.units,),
            initializer=chrono_init_cri(self.t_max, self.units, 1),
            trainable=True,
        )
        self.bias_i = self.add_weight(
            name="bias_i",
            shape=(self.units,),
            initializer=chrono_init_cri(self.t_max, self.units, 2),
            trainable=True,
        )
        self.bias_g = self.add_weight(
            name="bias_g",
            shape=(self.units,),
            initializer="zeros",
            trainable=True,
        )
        self.bias_o = self.add_weight(
            name="bias_o",
            shape=(self.units,),
            initializer=mchrono_init_cri(self.t_max, self.units),
            trainable=True,
        )

        self.ln_gates.build((None, 4 * self.units))
        self.ln_out.build((None, self.units))

        super().build(input_shape)

    def call(self, x, states):
        h = states[0]
        c = states[1]

        xh = tf.concat([x, h], axis=1)  
        z  = tf.matmul(xh, self.W)       

        z  = self.ln_gates(z)

        z_f, z_i, z_g, z_o = tf.split(z, num_or_size_splits=4, axis=1)

        z_f = tf.nn.bias_add(z_f, self.bias_f)
        z_i = tf.nn.bias_add(z_i, self.bias_i)
        z_g = tf.nn.bias_add(z_g, self.bias_g)
        z_o = tf.nn.bias_add(z_o, self.bias_o)

        f = tf.sigmoid(z_f)
        i = tf.sigmoid(z_i)
        g = tf.tanh(z_g)
        o = tf.sigmoid(z_o)

        c = f * c + i * g
        h = o * tf.tanh(c)

        out = self.ln_out(h)
        return out, [h, c]

In [ ]:
# works fine - separately by gates
"""
class CILN(tf.keras.layers.Layer):
    def __init__(self, units, t_max, **kwargs):
        self.units = units
        self.state_size = [units, units]
        self.t_max = t_max
        super(CILN, self).__init__(**kwargs)

    def build(self, input_shape):
        input_dim = input_shape[-1]

        self.bias_f = self.add_weight(
            name="bias_f",
            shape=(self.units,),
            initializer=chrono_init_cri(self.t_max, self.units, 1),
            trainable=True,
        )
        self.W_f = self.add_weight(
            name="W_fx",
            shape=(input_dim + self.units, self.units),
            initializer="glorot_uniform",
            trainable=True,
        )

        self.bias_i = self.add_weight(
            name="bias_i",
            shape=(self.units,),
            initializer=chrono_init_cri(self.t_max, self.units, 2),
            trainable=True,
        )
        self.W_i = self.add_weight(
            name="W_ix",
            shape=(input_dim + self.units, self.units),
            initializer="glorot_uniform",
            trainable=True,
        )

        self.bias_g = self.add_weight(
            name="bias_g",
            shape=(self.units,),
            initializer="zeros",
            trainable=True,
        )
        self.W_g = self.add_weight(
            name="W_gx",
            shape=(input_dim + self.units, self.units),
            initializer="glorot_uniform",
            trainable=True,
        )

        self.bias_o = self.add_weight(
            name="bias_o",
            shape=(self.units,),
            initializer=mchrono_init_cri(self.t_max, self.units),
            trainable=True,
        )
        self.W_o = self.add_weight(
            name="W_ox",
            shape=(input_dim + self.units, self.units),
            initializer="glorot_uniform",
            trainable=True,
        )

        self.LN = tf.keras.layers.LayerNormalization(center=False, scale=True, name="LN")
        self.LN.build((None, self.units))

        self.built = True

    def call(self, inputs, states):
        h_prev, c_prev = states
        xh = tf.concat([inputs, h_prev], axis=-1)

        f = tf.nn.bias_add(self.LN(tf.matmul(xh, self.W_f)), self.bias_f)
        i = tf.nn.bias_add(self.LN(tf.matmul(xh, self.W_i)), self.bias_i)
        g = tf.nn.bias_add(self.LN(tf.matmul(xh, self.W_g)), self.bias_g)

        c = tf.nn.sigmoid(f) * c_prev + tf.nn.sigmoid(i) * tf.nn.tanh(g)

        o = tf.nn.bias_add(self.LN(tf.matmul(xh, self.W_o)), self.bias_o)
        h = tf.nn.sigmoid(o) * tf.nn.tanh(c)

        return self.LN(h), [h, c]
        """

In [6]:
seq_len=200

In [7]:
x_train, y_train, x_valid, y_valid, x_test, y_test = get_add(n_data=150000, seq_len=seq_len)

In [8]:
inputs = Input(shape=(seq_len, 2))
cell = CILN(units=128, t_max=seq_len)
seq_output = RNN(cell, return_sequences=False)(inputs)
outputs = Dense(y_train.shape[-1])(seq_output)

model = Model(inputs=inputs, outputs=outputs)

model.compile(
    loss=tf.keras.losses.MeanSquaredError(),
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    metrics=[tf.keras.metrics.MeanSquaredError(name="mse")]
)

In [9]:
from keras.callbacks import ModelCheckpoint

In [10]:
best_model_path = "/content/best_model.keras"
model_checkpoint = ModelCheckpoint(filepath=best_model_path, monitor='val_mse', save_best_only=True, mode='min',  save_weights_only=False)

_ = model.fit(
    x_train, y_train,
    batch_size=50,
    epochs=5,
    validation_data=(x_valid, y_valid),
    callbacks=[model_checkpoint]
)

Epoch 1/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 88s 40ms/step - loss: 0.0426 - mse: 0.0426 - val_loss: 1.9306e-04 - val_mse: 1.9306e-04
Epoch 2/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 78s 39ms/step - loss: 1.7502e-04 - mse: 1.7502e-04 - val_loss: 1.0349e-04 - val_mse: 1.0349e-04
Epoch 3/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 77s 39ms/step - loss: 1.1023e-04 - mse: 1.1023e-04 - val_loss: 4.6971e-05 - val_mse: 4.6971e-05
Epoch 4/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 82s 39ms/step - loss: 7.8544e-05 - mse: 7.8544e-05 - val_loss: 6.4570e-05 - val_mse: 6.4570e-05
Epoch 5/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 82s 39ms/step - loss: 7.2683e-05 - mse: 7.2683e-05 - val_loss: 3.0581e-05 - val_mse: 3.0581e-05


In [11]:
from keras.models import load_model

In [12]:
best_model = load_model(best_model_path, custom_objects={"CILN": CILN})

test_loss, test_mse = best_model.evaluate(x_test, y_test)
print("Test mse:", test_mse)

1094/1094 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - loss: 3.1542e-05 - mse: 3.1542e-05
Test mse: 3.083789488300681e-05
